In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer  # used for filling missing values with mean
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder


In [3]:
df=pd.read_csv('dataset/covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


here `age` is numerical datatype, `gender, city` is nominal datatype `cough`  is ordinal datatype

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [6]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

here we can see that `fever `  has the null values hennce we have to fix it usign   `simple imputer`

In [7]:
df.head(2)

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes


In [8]:
# train test split

from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(df.drop('has_covid',axis=1),
                                               df['has_covid'],
                                               test_size=0.2,
                                               random_state=1
                                               )

x_train

,age,gender,fever,cough,city
2,42,Male,101.0,Mild,Delhi
73,34,Male,98.0,Strong,Kolkata
97,20,Female,101.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
19,42,Female,NaN,Strong,Bangalore
...,...,...,...,...,...
75,5,Male,102.0,Mild,Kolkata
9,64,Female,101.0,Mild,Delhi
72,83,Female,101.0,Mild,Kolkata
12,25,Female,99.0,Strong,Kolkata


# Aam Zindagi

In [9]:
    # handle missing value from fever

impute=SimpleImputer()

x_train_fever=impute.fit_transform(x_train[['fever']])

    # for test value of fever

x_test_fever=impute.fit_transform(x_test[['fever']])

In [10]:
np.isnan(x_train_fever).sum(),np.isnan(x_test_fever).sum()

(np.int64(0), np.int64(0))

as we can seee that there are no missing values


In [11]:
# now apply nominal encoding in city and gender

ohe=OneHotEncoder(drop='first')

x_train_gender_city=ohe.fit_transform(x_train[['gender','city']]).toarray()

# for test data

x_test_gender_city=ohe.fit_transform(x_test[['gender','city']]).toarray()

In [12]:
# now apply ordinal encoding in    `cough` 

oe=OrdinalEncoder(categories=[['Mild','Strong']])

x_train_cough=oe.fit_transform(x_train[['cough']])

# for xtest

x_test_cough=oe.fit_transform(x_test[['cough']])





In [13]:
# extract age 

x_train_age=x_train['age'].values # values for aray

# for test

x_test_age=x_test['age'].values

x_test_age.shape,x_train_age.shape


((20,), (80,))

In [14]:
x_train_age=x_train_age.reshape(-1,1)
x_test_age=x_test_age.reshape(-1,1)

In [15]:
x_train_age.shape

(80, 1)

In [16]:
## now concate all

x_train_transformed=np.concatenate((x_train_age,x_train_fever,x_train_gender_city,x_train_cough),axis=1)

# also test data

x_test_trasformed=np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)


x_train_transformed.shape

(80, 7)

In [17]:
## concacatenae need 2d array

# Mentos Zindagi

In [30]:
from sklearn.compose import ColumnTransformer
    #we have to pass two parameter inside the columntransformer ()  1. tansoformer and 2. remainder
coltrans=ColumnTransformer(transformers=[

                            ('tnf1',SimpleImputer(),['fever']),
                            ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
                            ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])

                            ]
                            ,remainder='passthrough')  # in remainder we can pass two variable 1. drop for droping the column which not participate in transformation , 
#2.'passthrough' keeping the remainder  column but dont apply any transformation 



# inside transformer , we have to do 3 types of transformer so we use 3 tuple in each of tuple we pass any tranformation name , then tranaformorfer class and then we pass the column on which the tranaformation should be applied

In [28]:
coltrans.fit_transform(x_train).shape


(80, 7)

In [29]:
coltrans.fit_transform(x_test).shape

(20, 7)